[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [LightGBM and XGBoost, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)

# Imbalanced Classes


## What you will be able to do

Judge a classifier for an event that almost never happens by its precision and recall, give the rare
class more weight in both libraries, and choose the probability at which to warn on stations the
model never trained on.

## The idea

### The problem

A road service salts every road in a district when the night will fall below −15 degrees. In the
station-days table that hard frost happens on 240 of the 15,000 training days, about 1.6% of them,
and on 38 of the 2,500 validation days. A forecast that never warned would be right on 98.5% of the
validation days and would miss every one of the nights the service exists for.

So accuracy cannot be the score, and the usual threshold may not be the right one either. A model
that has seen 240 hard-frost days and 14,760 others learns to say no with confidence, and a day has
to look very like the hard-frost days it has seen before its probability passes 0.5. The service
would rather salt a few roads that stay clear than leave one icy, and where to warn is a choice the
service makes with the model's probabilities.

### A formal definition

> A classification problem has **imbalanced classes** when one class is much rarer than the other.
> For the rare class, **precision** is the share of the days a model flags that really are that
> class, and **recall** is the share of that class's days the model flags. The **F1 score** is the
> harmonic mean of the two, which is high only when both are.

### Why it works that way

Log loss and accuracy are averages over every day, and nearly every day is an easy no, so a model
can score well on both while it flags few hard frosts. A weight on the rare class and a lower
threshold both shift that. The weight, `scale_pos_weight` in both libraries and LightGBM's
`class_weight="balanced"`, makes each hard-frost day count as many days in the loss, so the trees work harder to raise their
probabilities. A threshold below 0.5 flags more days from the same probabilities, trading precision
for recall. A threshold leaves the model's order of the days as it is, which average precision and
AUC measure over every threshold at once, and a weight trains a different model, so it can move those
scores either way.

A threshold is a setting like any other, and it has to be chosen on stations the model did not train
on. A boosted model fits its training days so closely that their probabilities are near 0 or 1, and a
threshold chosen there would say nothing about new stations. The validation stations choose it, and
the test stations measure the result once.

### Where this shows up

The **Classification** notebook introduced the probabilities and the scores used here, the **Tuning**
notebook chooses other settings by cross-validation on folds of whole training stations, and the
**A Frost Forecast** notebook chooses its models on the validation stations and measures them once on
the test stations.

### What this notebook covers

- hard frost, and how rare it is on each set of stations
- accuracy against precision, recall and F1
- the plain models at 0.5, and average precision
- precision and recall at lower thresholds
- weights for the rare class in both libraries
- a threshold chosen on the validation stations and measured once on the test stations
- three mistakes with rare classes

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
import numpy as np
from sklearn.metrics import accuracy_score, recall_score

actual = np.array([0] * 98 + [1] * 2)
always_no = np.zeros(100, dtype=int)
print(accuracy_score(actual, always_no), recall_score(actual, always_no))
```

```
0.98 0.0
```

Two days in a hundred are hard frost, and a forecast that always says no is right on 98 of them. Its
recall, the share of the two hard-frost days it caught, is 0.

## Setup

Two libraries for the table, the two boosting libraries, and the scores for a rare class.

- `numpy` and `pandas` build the table and the labels
- `lightgbm` and `xgboost` are LightGBM and XGBoost, and the cell prints their versions
- `accuracy_score`, `precision_score`, `recall_score` and `f1_score` score a model's labels, and
  `average_precision_score` and `roc_auc_score` its probabilities
- `warnings` and `UndefinedMetricWarning` catch the warning in the Common errors

`station_days` makes 20,000 rows of daily weather at 400 weather stations in Norway, the table of the
**Why Gradient Boosting** notebook. The stations and their days are made up, and a generator started
from a fixed seed draws them, so the table is the same on every run. `FEATURES` names the eight
columns a model is given, and the cell splits the stations into 300 for training, 50 for validation and
50 for testing, keeping all of a station's days on one side. This notebook trains on the training
stations, chooses on the validation stations, and measures the test stations once.

In [1]:
import warnings

import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from sklearn.exceptions import UndefinedMetricWarning
from sklearn.metrics import (accuracy_score, average_precision_score, f1_score, precision_score, recall_score,
                             roc_auc_score)

TERRAINS = ["coastal", "fjord", "inland", "valley", "mountain"]


def station_days(n_stations=400, days=50, seed=2026):
    """Made-up daily minimum temperatures at weather stations in Norway, one row per station and day."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n_stations)
    coast_km = np.minimum(rng.exponential(30, n_stations), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n_stations) + 2.5 * coast_km, 0, 1600)
    terrain = rng.choice(TERRAINS, n_stations, p=[0.2, 0.2, 0.25, 0.2, 0.15])
    has_snow_sensor = rng.uniform(size=n_stations) < 0.7
    offset = rng.normal(0, 0.8, n_stations)
    station = np.repeat(np.arange(n_stations), days)
    n = n_stations * days
    day = rng.integers(1, 366, n)
    cloud = np.clip(rng.beta(1.2, 1.0, n), 0, 1)
    wind = rng.gamma(2.0, 2.0, n)
    season = np.cos(2 * np.pi * (day - 20) / 365)
    lat, elev, coast = latitude[station], elevation[station], coast_km[station]
    snow = np.where(season > 0.2, np.clip(rng.normal(30 * season + 0.03 * elev + 2 * (lat - 58), 15), 0, None), 0.0)
    shelter = pd.Series(terrain[station]).map({"coastal": 0.4, "fjord": 1.0, "inland": 1.2, "valley": 1.7, "mountain": 0.8}).to_numpy()
    mean = 8 - 0.55 * (lat - 58) - 0.006 * elev - 9 * season + 3.5 * np.exp(-coast / 40) * season
    cooling = 7 * (1 - cloud) * np.exp(-wind / 3) * (0.4 + (season + 1) / 2) * shelter
    tmin = mean - 2.5 - cooling - 2.0 * (snow > 10) * (1 - cloud) + offset[station] + rng.normal(0, 1.2, n)
    snow_cm = np.where(has_snow_sensor[station], snow.round(0), np.nan)
    return pd.DataFrame({
        "station": [f"ST{s + 1:04d}" for s in station],
        "latitude": lat.round(2), "elevation_m": elev.round(0).astype(int), "coast_km": coast.round(1),
        "terrain": pd.Categorical(terrain[station], categories=TERRAINS),
        "day_of_year": day, "cloud_cover": cloud.round(2), "wind_ms": wind.round(1), "snow_cm": snow_cm,
        "tmin": tmin.round(1),
    })


FEATURES = ["latitude", "elevation_m", "coast_km", "terrain", "day_of_year", "cloud_cover", "wind_ms", "snow_cm"]
days = station_days()
stations = days["station"].unique()
order = np.random.default_rng(0).permutation(len(stations))
train = days[days["station"].isin(stations[order[:300]])]
valid = days[days["station"].isin(stations[order[300:350]])]
test = days[days["station"].isin(stations[order[350:]])]

print("lightgbm", lgb.__version__, "| xgboost", xgb.__version__, "|", len(train), "training rows,",
      len(valid), "validation rows,", len(test), "test rows")

lightgbm 4.6.0 | xgboost 3.4.1 | 15000 training rows, 2500 validation rows, 2500 test rows


## Worked examples

### Hard frost on each set of stations

A day has hard frost when its lowest temperature is below −15. The labels are 1 for hard frost and 0
for any other day.

In [2]:
hard_train = (train["tmin"] < -15).astype(int)
hard_valid = (valid["tmin"] < -15).astype(int)
hard_test = (test["tmin"] < -15).astype(int)
for name, labels in (("training", hard_train), ("validation", hard_valid), ("test", hard_test)):
    print(f"{name:10} {labels.sum():3} of {len(labels)} days, {labels.mean():.4f}")

training   240 of 15000 days, 0.0160
validation  38 of 2500 days, 0.0152
test        38 of 2500 days, 0.0152


Hard frost falls on 240 training days, 1.6% of them, and on 38 days in each of the validation and test
sets. A recall measured on 38 days moves by about 0.026 for every day a model catches or misses, which
is worth keeping in mind in every comparison below.

### Accuracy against precision and recall

A forecast that never warns, scored on the validation stations. `zero_division=0` tells
`precision_score` to report 0 when a model flags no day at all, which the Common errors explain.

In [3]:
never = np.zeros(len(hard_valid), dtype=int)
print("accuracy", round(accuracy_score(hard_valid, never), 4), "| precision",
      precision_score(hard_valid, never, zero_division=0), "| recall", recall_score(hard_valid, never))

accuracy 0.9848 | precision 0.0 | recall 0.0


The forecast that never warns is right on 98.48% of the validation days, and its recall is 0, since it
flags none of the 38 hard frosts. With no day flagged, its precision is 0 divided by 0, which
`zero_division=0` reports as 0.

### The plain models at 0.5

`report` turns probabilities into flags at a threshold and prints how many days were flagged, the
three scores of those flags, and the two scores of the probabilities themselves.

In [4]:
def report(name, labels, probability, threshold=0.5):
    flagged = (probability > threshold).astype(int)
    print(f"{name:20} flags {flagged.sum():3} | precision {precision_score(labels, flagged, zero_division=0):.3f} "
          f"recall {recall_score(labels, flagged):.3f} F1 {f1_score(labels, flagged):.3f} | "
          f"average precision {average_precision_score(labels, probability):.3f} AUC {roc_auc_score(labels, probability):.3f}")

lightgbm_plain = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, random_state=0, verbose=-1)
lightgbm_plain.fit(train[FEATURES], hard_train)
xgboost_plain = xgb.XGBClassifier(n_estimators=300, learning_rate=0.05, random_state=0)
xgboost_plain.fit(train[FEATURES], hard_train)

report("LightGBM", hard_valid, lightgbm_plain.predict_proba(valid[FEATURES])[:, 1])
report("XGBoost", hard_valid, xgboost_plain.predict_proba(valid[FEATURES])[:, 1])

LightGBM             flags  14 | precision 0.643 recall 0.237 F1 0.346 | average precision 0.599 AUC 0.988
XGBoost              flags  16 | precision 0.688 recall 0.289 F1 0.407 | average precision 0.667 AUC 0.989


At 0.5 LightGBM flags 14 validation days and XGBoost 16. Most of what they flag is hard frost, a
precision of 0.643 and 0.688, and they find about a quarter of the 38 days, a recall of 0.237 and
0.289. Average precision, the area under the curve of precision against recall over every threshold,
is 0.599 and 0.667, and the AUC is 0.988 and 0.989, so both models order the days well, and the 0.5
threshold uses little of that order.

### Lower thresholds

The same LightGBM probabilities, flagged at thresholds from 0.5 down to 0.02.

In [5]:
lightgbm_valid = lightgbm_plain.predict_proba(valid[FEATURES])[:, 1]
for threshold in (0.5, 0.3, 0.2, 0.1, 0.05, 0.02):
    report(f"LightGBM at {threshold}", hard_valid, lightgbm_valid, threshold)

LightGBM at 0.5      flags  14 | precision 0.643 recall 0.237 F1 0.346 | average precision 0.599 AUC 0.988
LightGBM at 0.3      flags  17 | precision 0.706 recall 0.316 F1 0.436 | average precision 0.599 AUC 0.988
LightGBM at 0.2      flags  22 | precision 0.636 recall 0.368 F1 0.467 | average precision 0.599 AUC 0.988
LightGBM at 0.1      flags  24 | precision 0.583 recall 0.368 F1 0.452 | average precision 0.599 AUC 0.988
LightGBM at 0.05     flags  33 | precision 0.545 recall 0.474 F1 0.507 | average precision 0.599 AUC 0.988
LightGBM at 0.02     flags  45 | precision 0.556 recall 0.658 F1 0.602 | average precision 0.599 AUC 0.988


Lowering the threshold flags more days, from 14 at 0.5 to 45 at 0.02, and recall rises from 0.237 to
0.658. Precision stays between 0.545 and 0.706, so on these stations the F1 score is highest at 0.02.
Average precision and AUC are the same on every line, since they score the probabilities, which a
threshold does not change.

### Weights for the rare class

`scale_pos_weight` multiplies the weight of every day labeled 1, and the usual value is the number of
days labeled 0 for each day labeled 1. LightGBM's `class_weight="balanced"` gives each class a weight
that makes the two weigh the same in total.

In [6]:
ratio = (hard_train == 0).sum() / hard_train.sum()
print("days labeled 0 for each day labeled 1:", round(ratio, 1))

lightgbm_weighted = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, random_state=0, verbose=-1,
                                       scale_pos_weight=ratio)
lightgbm_balanced = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, random_state=0, verbose=-1,
                                       class_weight="balanced")
xgboost_weighted = xgb.XGBClassifier(n_estimators=300, learning_rate=0.05, random_state=0, scale_pos_weight=ratio)
for name, model in (("LightGBM weighted", lightgbm_weighted), ("LightGBM balanced", lightgbm_balanced),
                    ("XGBoost weighted", xgboost_weighted)):
    model.fit(train[FEATURES], hard_train)
    report(name, hard_valid, model.predict_proba(valid[FEATURES])[:, 1])

days labeled 0 for each day labeled 1: 61.5
LightGBM weighted    flags  25 | precision 0.600 recall 0.395 F1 0.476 | average precision 0.608 AUC 0.987
LightGBM balanced    flags  27 | precision 0.593 recall 0.421 F1 0.492 | average precision 0.575 AUC 0.985
XGBoost weighted     flags  24 | precision 0.708 recall 0.447 F1 0.548 | average precision 0.627 AUC 0.986


With a weight of 61.5 on each hard-frost day, LightGBM flags 25 days at 0.5 in place of 14 and
XGBoost 24 in place of 16, and recall rises to 0.395 and 0.447. Average precision moves both ways, up
from 0.599 to 0.608 for LightGBM with `scale_pos_weight`, down to 0.575 with `class_weight="balanced"`,
and down from 0.667 to 0.627 for XGBoost. On these stations the weights moved the trade at 0.5 toward
recall without ordering the days better, and with 38 hard-frost days, differences of this size are a
few days either way.

### Probabilities on the training stations

The cell prints the plain LightGBM model's mean probability on the hard-frost days and on the other
days, and its F1 at two thresholds, on the stations it trained on and on the validation stations.

In [7]:
for name, rows, labels in (("training", train, hard_train), ("validation", valid, hard_valid)):
    probability = lightgbm_plain.predict_proba(rows[FEATURES])[:, 1]
    print(f"{name:10} hard-frost days {probability[labels == 1].mean():.3f} | other days {probability[labels == 0].mean():.3f}"
          f" | F1 at 0.02 {f1_score(labels, (probability > 0.02).astype(int)):.3f}, at 0.5 "
          f"{f1_score(labels, (probability > 0.5).astype(int)):.3f}")

training   hard-frost days 0.991 | other days 0.000 | F1 at 0.02 0.974, at 0.5 1.000
validation hard-frost days 0.276 | other days 0.002 | F1 at 0.02 0.602, at 0.5 0.346


On the stations it trained on, the model gives the hard-frost days a mean probability of 0.991,
and a threshold of 0.5 scores an F1 of 1.000, above the 0.974 at 0.02. On the validation stations the
hard-frost days average 0.276, and the two thresholds change places, with 0.346 at 0.5 against 0.602
at 0.02. The trees fitted the training days almost exactly, so the training stations cannot show which
threshold suits new stations, and the next cell chooses one on the validation stations instead.

### A threshold chosen on the validation stations

`best_threshold` tries every threshold from 0.01 to 0.5 in steps of 0.01 on the validation stations
and keeps the one with the highest F1. The test stations are then scored once, at that threshold.

In [8]:
THRESHOLDS = np.round(np.arange(0.01, 0.51, 0.01), 2)

def best_threshold(labels, probability):
    scores = [f1_score(labels, (probability > threshold).astype(int)) for threshold in THRESHOLDS]
    return THRESHOLDS[int(np.argmax(scores))], max(scores)

for name, model in (("LightGBM", lightgbm_plain), ("XGBoost", xgboost_plain)):
    threshold, valid_f1 = best_threshold(hard_valid, model.predict_proba(valid[FEATURES])[:, 1])
    test_flags = (model.predict_proba(test[FEATURES])[:, 1] > threshold).astype(int)
    print(f"{name:8} threshold {threshold} | validation F1 {valid_f1:.3f} | test F1 {f1_score(hard_test, test_flags):.3f}")

LightGBM threshold 0.02 | validation F1 0.602 | test F1 0.578
XGBoost  threshold 0.09 | validation F1 0.667 | test F1 0.695


On the validation stations LightGBM's best threshold is 0.02, with an F1 of 0.602, and XGBoost's is
0.09, with 0.667. Measured once on the test stations, the two thresholds give F1 scores of 0.578 and
0.695, which are the estimates to report for these models. A test F1 can come out above or below the
validation F1, since each rests on 38 days.

## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/lightgbm-and-xgboost-deep-dive/11-imbalanced-classes-solutions.ipynb).

**1.** Print the precision, recall and F1 of the plain LightGBM model at a threshold of 0.1 on the validation stations, with `report`.

In [9]:
# your code here

**2.** Train an `LGBMClassifier` with `is_unbalance=True` in place of `scale_pos_weight`, and print its line of `report` on the validation stations.

In [10]:
# your code here

**3.** Choose a threshold for `xgboost_weighted` with `best_threshold` on the validation stations, and print its F1 on the test stations once.

In [11]:
# your code here

**4.** Make labels for a lowest temperature below −12 under new names, leaving `hard_train` and `hard_valid` as they are, and print the share of training days they cover and the average precision on the validation stations of a plain LightGBM model trained on them.

In [12]:
# your code here

**5.** For `xgboost_plain`, print the mean probability it gives the validation days that had hard frost and the days that did not.

In [13]:
# your code here

## Common errors

### `UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples.`

The cell makes the validation labels and the forecast that never warns, scores it without
`zero_division`, records the warning and prints its category and first sentence.

In [14]:
hard_frost_valid = (valid["tmin"] < -15).astype(int)
never_warns = np.zeros(len(hard_frost_valid), dtype=int)
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always", UndefinedMetricWarning)
    print("precision:", precision_score(hard_frost_valid, never_warns))
for warning in caught:
    print(f"{warning.category.__name__}: {str(warning.message).split('. ')[0]}.")

precision: 0.0


Precision is the number of flagged days that had hard frost divided by the number of flagged days, and
with none flagged that is 0 divided by 0. scikit-learn reports 0.0 and warns, and a loop over
thresholds prints the warning again for every threshold that no day's probability passes. Pass `zero_division=0`
when a model may flag nothing, as `report` does, and read a precision of 0 with no days flagged as no
answer.

In [15]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always", UndefinedMetricWarning)
    print("precision:", precision_score(hard_frost_valid, never_warns, zero_division=0))
print("warnings:", len(caught))

precision: 0.0
warnings: 0


With `zero_division=0` the precision is 0.0 again, and the cell records no warning.

### `LightGBMError: Cannot set is_unbalance and scale_pos_weight at the same time`

In [16]:
hard_frost_train = (train["tmin"] < -15).astype(int)
lgb.LGBMClassifier(n_estimators=10, verbose=-1, is_unbalance=True, scale_pos_weight=ratio).fit(train[FEATURES], hard_frost_train)

[LightGBM] [Fatal] Cannot set is_unbalance and scale_pos_weight at the same time


LightGBMError: Cannot set is_unbalance and scale_pos_weight at the same time

`is_unbalance=True` tells LightGBM to work out a weight for the rare class itself, and
`scale_pos_weight` gives it one, so when both are set, LightGBM stops, printing its `[LightGBM] [Fatal]`
line before Python raises the error. The next cell keeps `scale_pos_weight` alone.

In [17]:
one_weight = lgb.LGBMClassifier(n_estimators=10, verbose=-1, scale_pos_weight=ratio).fit(train[FEATURES], hard_frost_train)
print(one_weight.booster_.num_trees(), "trees")

10 trees


With one of the two settings, the model trains its 10 trees.

### No error, and a threshold chosen on the test stations

The cell makes the labels and trains the plain LightGBM model again, then chooses a threshold with
`best_threshold` once on the test stations and once on the validation stations, and scores both on the
test stations.

In [18]:
hard_frost_train = (train["tmin"] < -15).astype(int)
hard_frost_valid = (valid["tmin"] < -15).astype(int)
hard_frost_test = (test["tmin"] < -15).astype(int)
plain_again = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, random_state=0, verbose=-1)
plain_again.fit(train[FEATURES], hard_frost_train)
valid_probability = plain_again.predict_proba(valid[FEATURES])[:, 1]
test_probability = plain_again.predict_proba(test[FEATURES])[:, 1]
chosen_on_test, test_f1_on_test = best_threshold(hard_frost_test, test_probability)
chosen_on_valid, _ = best_threshold(hard_frost_valid, valid_probability)
print("chosen on the test stations:", chosen_on_test, "| test F1", round(test_f1_on_test, 3))
print("chosen on the validation stations:", chosen_on_valid, "| test F1",
      round(f1_score(hard_frost_test, (test_probability > chosen_on_valid).astype(int)), 3))

chosen on the test stations: 0.06 | test F1 0.667
chosen on the validation stations: 0.02 | test F1 0.578


Chosen on the test stations, the threshold is 0.06 and the test F1 0.667, against 0.578 for the
threshold the validation stations chose. Both scores come from the same model and the same 38 test
days, and the first is higher because the test days chose it, so it overstates what the model would
score on stations it has not seen. Choose on the validation stations, and report the test score of
that choice.

## Recap

- You can score a model of a rare class with precision, recall and F1, and its probabilities with
  average precision.
- You can give the rare class more weight with `scale_pos_weight` in both libraries, or with
  LightGBM's `class_weight="balanced"` or `is_unbalance`.
- You can lower the threshold to trade precision for recall, and read the trade at each threshold.
- You can choose a threshold on the validation stations and measure it once on the test stations.

## What is next

The **Tuning** notebook chooses the settings of a model by cross-validation on folds of whole training
stations, scoring each candidate on stations it did not train on and keeping the test stations out of
the search.

---

&#8592; **Previous:** [Classification](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/lightgbm-and-xgboost-deep-dive/10-classification.ipynb)  &nbsp;·&nbsp;  [LightGBM and XGBoost, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Tuning](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/lightgbm-and-xgboost-deep-dive/12-tuning.ipynb) &#8594;
